# Sampling vs MCTS: Phase 2 (Colab)

**Plan A pgx-native pipeline.** Run-order:

1. GPU check + Drive mount
2. Clone repo + install
3. wandb login
4. **Setup smoke** (~1 min) — verify install end-to-end
5. **Per-iter bench** (~5-30 min) — measure real throughput, calibrate config
6. **Mini Phase 2** (~30 min - 4 h per arm) — verify training improves
7. **Full Phase 2** (~10-30 h per arm) — doc-spec runs

Recommended GPU: **G4 (RTX PRO 6000 Blackwell, 96 GB)** for ~2× A100 throughput at similar unit/h. Falls back gracefully to A100; L4 is too slow for steps 5+.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/sampling-chess'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
print('Drive base:', DRIVE_BASE)

## 2. Clone + install

In [ ]:
%cd /content
![ -d sampling-chess ] || git clone https://github.com/hongttisme/sampling-chess.git
%cd sampling-chess
!git pull --ff-only

In [ ]:
# Editable install + ml extras (jax/flax/optax/mctx/wandb) + pgx separately.
!pip install -q -e ".[ml]" pgx
# Stockfish for eval opponent.
!apt-get install -qq -y stockfish && which stockfish

In [ ]:
# Verify the GPU stack is wired through.
import jax, flax, optax, mctx, pgx
print(f'jax {jax.__version__}  flax {flax.__version__}  optax {optax.__version__}')
print(f'mctx {mctx.__version__}  pgx {pgx.__version__}')
print(f'devices: {jax.devices()}')
print(f'backend: {jax.default_backend()}')

## 3. wandb login

Paste your API key from https://wandb.ai/authorize when prompted.

In [ ]:
!wandb login

## 4. Setup smoke (~1 min)

Same args as the local CPU smoke (117s on a 12-core box). Should run in **seconds** on any GPU. If this errors, fix before going further.

In [ ]:
!python scripts/09_phase2_full_smoke.py --arm b --iters 2 --games-per-iter 2 \
    --train-steps 15 --batch 4 --max-plies 8 --eval-every 1 --eval-games 2

## 5. Per-iter bench (~5-30 min)

Single iter at near-real config; the printed wall-clock per iter is what you multiply by `n_iterations` for a budget. **Read the iter-line wall-clock** to decide whether step 6/7's defaults are sensible.

Rough projection:
- iter time ~5 min  → full 50-iter run ~4 h
- iter time ~15 min → full 50-iter run ~12 h
- iter time ~30 min → full 50-iter run ~25 h

In [ ]:
# Arm B bench — usually slower than Arm A so worth profiling first.
!python scripts/09_phase2_full_smoke.py --arm b --iters 1 --games-per-iter 16 \
    --train-steps 100 --batch 64 --max-plies 200 --eval-every 0 --wandb \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/bench_arm_b.log

In [ ]:
# Arm A bench (mctx).
!python scripts/09_phase2_full_smoke.py --arm a --iters 1 --games-per-iter 16 \
    --train-steps 100 --batch 64 --max-plies 200 --eval-every 0 --wandb \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/bench_arm_a.log

## 6. Mini Phase 2 (both arms, ~30 min - 4 h each)

After step 5 you know roughly what an iter costs. The defaults below assume ~5-15 min/iter on G4. Adjust `--iters`, `--games-per-iter`, `--train-steps` if your bench differs.

**Goal**: see loss curve actually decrease across iterations + first signs of eval Elo above random.

In [ ]:
!python scripts/09_phase2_full_smoke.py --arm b --iters 10 --games-per-iter 32 \
    --train-steps 200 --batch 128 --max-plies 200 \
    --eval-every 5 --eval-games 8 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_b \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/mini_arm_b.log

In [ ]:
!python scripts/09_phase2_full_smoke.py --arm a --iters 10 --games-per-iter 32 \
    --train-steps 200 --batch 128 --max-plies 200 \
    --eval-every 5 --eval-games 8 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_a \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/mini_arm_a.log

## 7. Full Phase 2 (~10-30 h per arm)

doc-spec config: 50 iters × 256 games per iter × 1000 train steps × eval every 5 iters at multiple skill levels. Run each arm in **separate sessions** (Colab disconnects every 12 h on Pro / 24 h on Pro+; checkpoints below are written to Drive so you can resume if needed).

If your bench in step 5 says iter > 30 min, halve `--games-per-iter` to 128 first.

In [ ]:
# Full Arm B run. --ckpt-dir auto-resumes from latest.pkl if Colab disconnects.
!python scripts/09_phase2_full_smoke.py --arm b --iters 50 --games-per-iter 256 \
    --train-steps 1000 --batch 1024 --max-plies 400 \
    --eval-every 5 --eval-games 20 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/full_arm_b \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/full_arm_b.log

In [ ]:
# Full Arm A run. --ckpt-dir auto-resumes from latest.pkl if Colab disconnects.
!python scripts/09_phase2_full_smoke.py --arm a --iters 50 --games-per-iter 256 \
    --train-steps 1000 --batch 1024 --max-plies 400 \
    --eval-every 5 --eval-games 20 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/full_arm_a \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/full_arm_a.log

## Notes

- **Checkpoints**: `--ckpt-dir DIR` saves `iter_NNNNN.pkl` + `latest.pkl` after each iter. **Resume is automatic** if `latest.pkl` exists in the dir; pass `--no-resume` to start fresh. Buffer is NOT persisted (refills via self-play, takes ~2 iters to refill).
- **Multi-skill eval**: `--eval-skill` currently takes one int. To eval at {0, 3, 5}, run separately or add a `--eval-skills 0,3,5` flag.
- **Phase 3 ablations**: vary K, k_plies, beta on Arm B — same script, different `make_arm_b_op_builder` args. The CLI exposes these one rename away (TODO: surface as `--K --k-plies --beta`).

Per-iter wall-clock from your bench in step 5 is the source of truth for budgeting; don't trust the rough numbers in this notebook.